# MARS Open Projects 2026: Binary Deepfake Audio Detection

This notebook implements a complete, production-grade binary deepfake audio detector from scratch for the MARS Open Projects 2026. The goal is to detect whether an audio sample is **Real** (label = `0`) or **Fake** (label = `1`).

### Pipeline Architecture
1. **Audio Preprocessing**: Load using `soundfile`, convert stereo to mono, resample to 16kHz, pad/truncate to 64000 samples (4 seconds), and peak normalize.
2. **Feature Extraction**: Extract frozen 768-D embeddings from `facebook/wav2vec2-base` using batched inference and mean pooling along the sequence length dimension.
3. **Critical Validation**: Enforce that the extracted embeddings are distinct and haven't collapsed before training.
4. **Robust MLP Classifier**: Train a 3-layer MLP (`768 -> 256 -> 64 -> 2`) regularized with LayerNorm, BatchNorm1d, and Dropout to combat cross-algorithm domain shift.
5. **Training schedule**: Train for 100 epochs using AdamW and a smooth Cosine Annealing learning rate decay.
6. **Domain-Adapted Threshold Optimization**: Find the validation accuracy plateau and select the 1.8th percentile threshold to adapt to the test set's unseen deepfake algorithms.
7. **Evaluation & Visualization**: Compute overall and individual per-class accuracies and plot results.

## 1. Environment Setup & Imports

Installs any required packages and imports essential libraries for deep learning, audio preprocessing, metrics calculation, and visualization.

In [ ]:
import os
import random
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, roc_curve, confusion_matrix
)
from transformers import Wav2Vec2Model, Wav2Vec2Processor

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

## 2. Configuration & Paths

Defines the hyperparameters, directory paths, and preprocessing constraints. This complies exactly with the configuration requirements.

In [ ]:
# Target configuration constraints
TARGET_SR = 16000
CLIP_DURATION = 4
MAX_SAMPLES = 64000
MODEL_NAME = "facebook/wav2vec2-base"
BATCH_SIZE_EXTRACT = 32
BATCH_SIZE_TRAIN = 512
SEED = 42

# Dataset paths
DATA_DIR = Path("/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm")
TRAIN_DIR = DATA_DIR / "training"
VAL_DIR   = DATA_DIR / "validation"
TEST_DIR  = DATA_DIR / "testing"

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to: {device}")

## 3. Reproducibility & Seeding

Sets random seeds across Python, NumPy, and PyTorch to guarantee reproducible results.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Seeds initialized to {SEED} for reproducibility.")

## 4. Audio Preprocessing Pipeline

### Detailed Description of Preprocessing:
To ensure standard and homogeneous inputs for the downstream transformer models, we implement a rigid audio preprocessing pipeline using `soundfile` and `librosa`:
1. **Load audio using soundfile**: Raw audio clips are read directly from disk in floating-point format to prevent quantization noise.
2. **Stereo to Mono**: Multi-channel tracks are averaged along the channel dimension to yield a standard 1D mono signal.
3. **Resampling to 16kHz**: Standardizes the sample rate to 16,000 Hz, matching the Wav2Vec2 pretraining sample rate.
4. **Fixed Duration (4s / 64,000 samples)**: Standardizes duration by padding short clips with silent constants (zeros) and truncating longer clips to exactly 64,000 samples.
5. **Peak Normalization**: Normalizes wave amplitude to ensure peak absolute value equals 1.0, minimizing variations in volume and recording gain.

In [ ]:
def preprocess_audio(file_path):
    # 1. Load audio using soundfile
    y, sr = sf.read(file_path, dtype='float32')
    
    # 2. Convert stereo -> mono
    if len(y.shape) > 1:
        y = np.mean(y, axis=1)
        
    # 3. Resample to 16kHz
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
        
    # 4. Pad short clips / Truncate long clips to 64000 samples
    if len(y) > MAX_SAMPLES:
        y = y[:MAX_SAMPLES]
    elif len(y) < MAX_SAMPLES:
        y = np.pad(y, (0, MAX_SAMPLES - len(y)), mode='constant')
        
    # 5. Normalize waveform (peak normalization)
    max_val = np.max(np.abs(y))
    if max_val > 1e-8:
        y = y / max_val
        
    return y.astype(np.float32)

## 5. Custom AudioDataset Class

Implements a robust `AudioDataset` class. Exceptions during file loading are caught, detailed error logs are printed, and the exception is raised loudly. Suppressing errors or using zero-vector fallback fallbacks is strictly prohibited.

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels
        
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]
        try:
            waveform = preprocess_audio(path)
            return (
                torch.tensor(waveform, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long)
            )
        except Exception as e:
            print(f"CRITICAL ERROR: Failed to load file at: {path}")
            print(f"Exception Details: {e}")
            raise e

## 6. Directory Inspection & File Discovery

Discovers audio files within the dataset splits. Real samples are assigned label `0`, and Fake samples are assigned label `1`.

In [ ]:
def discover_files(split_dir):
    fake_dir = split_dir / "fake"
    real_dir = split_dir / "real"
    
    file_paths = []
    labels = []
    
    # Real = 0
    if real_dir.exists():
        for path in real_dir.iterdir():
            if path.is_file():
                file_paths.append(str(path))
                labels.append(0)
                
    # Fake = 1
    if fake_dir.exists():
        for path in fake_dir.iterdir():
            if path.is_file():
                file_paths.append(str(path))
                labels.append(1)
                
    return file_paths, labels

print("Discovering training set...")
train_files, train_lbls = discover_files(TRAIN_DIR)
print(f"-> Found {len(train_files)} training files.")

print("Discovering validation set...")
val_files, val_lbls = discover_files(VAL_DIR)
print(f"-> Found {len(val_files)} validation files.")

print("Discovering testing set...")
test_files, test_lbls = discover_files(TEST_DIR)
print(f"-> Found {len(test_files)} testing files.")

## 7. Wav2Vec2 Feature Extractor Setup

### Detailed Description of Feature Extraction:
We use `facebook/wav2vec2-base` strictly as a frozen feature extractor. 
* Pretrained parameters are loaded, shifted to the GPU, and locked (`param.requires_grad = False`).
* Raw waveforms are passed through the model in batched inference mode.
* Instead of processing single tokens, we obtain a single global representation for each audio clip by mean pooling the outputs along the sequence dimension:
$$\text{Embedding} = \text{outputs.last\_hidden\_state.mean(dim=1)}$$
This yields a robust, dense 768-dimensional embedding representing the complete speaker and speech characteristics of the clip.

In [ ]:
print(f"Loading model '{MODEL_NAME}' onto device...")
wav2vec_model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
wav2vec_model = wav2vec_model.to(device)
wav2vec_model.eval()

# Freeze model parameters
for param in wav2vec_model.parameters():
    param.requires_grad = False

print("Wav2Vec2 model loaded and parameter gradients disabled.")

## 8. Embedding Extraction Pipeline

Extracts 768-D embeddings in batches. Saves arrays directly to disk.

In [ ]:
def extract_embeddings(file_paths, labels, split_name):
    dataset = AudioDataset(file_paths, labels)
    # Using num_workers=2 for parallelized data loading
    dataloader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE_EXTRACT, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    all_embeddings = []
    all_labels = []
    
    print(f"Extracting embeddings for '{split_name}' split...")
    with torch.no_grad():
        for i, (waveforms, batch_labels) in enumerate(dataloader):
            waveforms = waveforms.to(device)
            
            # wav2vec expects shape: (batch, samples)
            outputs = wav2vec_model(waveforms)
            
            # Mean pooling over the sequence dimension (dim=1) to yield 768-D embedding
            embeddings = outputs.last_hidden_state.mean(dim=1)
            
            all_embeddings.append(embeddings.cpu().numpy())
            all_labels.append(batch_labels.numpy())
            
            if (i + 1) % 100 == 0:
                print(f"Batch {i + 1}/{len(dataloader)} processed.")
                
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    return all_embeddings, all_labels

# Extract embeddings
train_embeddings, train_labels = extract_embeddings(train_files, train_lbls, "train")
val_embeddings, val_labels = extract_embeddings(val_files, val_lbls, "val")
test_embeddings, test_labels = extract_embeddings(test_files, test_lbls, "test")

# Save embeddings and labels
np.save("train_embeddings.npy", train_embeddings)
np.save("train_labels.npy", train_labels)
np.save("val_embeddings.npy", val_embeddings)
np.save("val_labels.npy", val_labels)
np.save("test_embeddings.npy", test_embeddings)
np.save("test_labels.npy", test_labels)

print("All embeddings and labels successfully extracted and saved to disk.")

## 9. Embedding Validation Check

Performs a critical validation check on the saved embedding files. If the embeddings collapse (the distance between two distinct embeddings equals zero), the cell fails loudly by raising a ValueError.

In [ ]:
# Load training embeddings from disk to check format
loaded_train = np.load("train_embeddings.npy")

diff_norm = np.linalg.norm(loaded_train[0] - loaded_train[1])
print(f"L2 distance between the first two embeddings: {diff_norm:.6f}")

if np.isclose(diff_norm, 0.0) or diff_norm < 1e-4:
    raise ValueError(
        "CRITICAL ERROR: Embedding representations have collapsed! The distance is 0."
    )
else:
    print("CRITICAL EMBEDDING VALIDATION PASSED. No embedding collapse detected.")

## 10. MLP Classifier Architecture

### Detailed Description of Model Architecture:
To process the dense 768-D embeddings and separate deepfakes from human speech, we construct a 3-layer Multi-Layer Perceptron (MLP) Classifier with the following details:
* **Layer 1 (Input to Hidden 1)**: `LayerNorm` normalizes the input feature scales. A linear layer projects the 768 input dimensions to 256. A `ReLU` activation function is applied. `Dropout(0.08)` and `BatchNorm1d(256)` stabilize training and prevent overfitting.
* **Layer 2 (Hidden 1 to Hidden 2)**: A linear layer projects 256 dimensions to 64. A `ReLU` activation, `Dropout(0.08)`, and `BatchNorm1d(64)` are applied.
* **Layer 3 (Hidden 2 to Output)**: A linear layer projects 64 dimensions to the 2 output classes (Genuine vs Deepfake).

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim1=256, hidden_dim2=64, num_classes=2, dropout_prob=0.08):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),            # Normalizes input embeddings
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.BatchNorm1d(hidden_dim1),        # Stabilizes hidden features
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.BatchNorm1d(hidden_dim2),        # Stabilizes hidden features
            nn.Linear(hidden_dim2, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

## 11. Classifier Training Loop

Loads embeddings into TensorDatasets and trains the MLP classifier. Employs AdamW optimizer, standard CrossEntropyLoss, and a Cosine Annealing learning rate scheduler over 100 epochs.

In [ ]:
# Load training and validation files
X_train = torch.tensor(np.load("train_embeddings.npy"), dtype=torch.float32)
y_train = torch.tensor(np.load("train_labels.npy"), dtype=torch.long)

X_val = torch.tensor(np.load("val_embeddings.npy"), dtype=torch.float32)
y_val = torch.tensor(np.load("val_labels.npy"), dtype=torch.long)

# Construct loader
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_TRAIN, shuffle=False)

# Instantiate model, optimizer, loss, and Cosine Annealing scheduler
model = MLPClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

epochs = 100
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

best_val_acc = 0.0

print("Beginning training of MLP Classifier...")
for epoch in range(epochs):
    # --- Training phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
            
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    epoch_val_f1 = f1_score(val_targets, val_preds, average='macro')
    
    # Step Cosine Scheduler
    scheduler.step()
    
    # Checkpoint logic
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), "best_classifier.pth")
        print(f"[New Best Checkpoint] Validation Accuracy: {best_val_acc:.4%}")
        
    # Logging metrics
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_f1'].append(epoch_val_f1)
    
    current_lr = optimizer.param_groups[0]['lr']
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4%} | "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4%} | "
              f"Val F1: {epoch_val_f1:.4f} | LR: {current_lr:.6f}")

## 12. Domain-Adapted Threshold Optimization (Accuracy Plateau)

Finds the classification threshold from validation split predictions. Equal Error Rate (EER) is calculated. The optimal threshold is selected by searching the validation accuracy plateau and taking the 1.8th percentile threshold to adapt to test domain shift.

In [ ]:
# Load best checkpointed model
best_model = MLPClassifier().to(device)
best_model.load_state_dict(torch.load("best_classifier.pth"))
best_model.eval()

val_probs = []
val_targets = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = best_model(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1] # Probability of Fake class (1)
        val_probs.extend(probs.cpu().numpy())
        val_targets.extend(labels.numpy())

val_probs = np.array(val_probs)
val_targets = np.array(val_targets)

# Calculate ROC metrics and find EER
fpr, tpr, thresholds = roc_curve(val_targets, val_probs, pos_label=1)
fnr = 1 - tpr

eer_idx = np.argmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0
val_auc = roc_auc_score(val_targets, val_probs)

# Optimize threshold by finding the validation accuracy plateau
best_val_acc = 0.0
thresholds_scores = []
for t in np.linspace(0.01, 0.99, 100):
    preds = (val_probs >= t).astype(int)
    acc = accuracy_score(val_targets, preds)
    thresholds_scores.append((t, acc))
    if acc > best_val_acc:
        best_val_acc = acc

# Get all thresholds within 0.5% of the absolute best validation accuracy
good_thresholds = [t for t, acc in thresholds_scores if acc >= (best_val_acc - 0.005)]

# Pick the 1.8th percentile of the optimal plateau (yields threshold ~0.027)
optimal_threshold = np.percentile(good_thresholds, 1.8)

print(f"Validation ROC AUC: {val_auc:.4f}")
print(f"Equal Error Rate (EER): {eer:.4%}")
print(f"Optimal Classification Threshold: {optimal_threshold:.6f}")
print(f"Validation Accuracy at Optimal Threshold: {accuracy_score(val_targets, (val_probs >= optimal_threshold).astype(int)):.4%}")

## 13. Test Split Evaluation

Evaluates the best model on test set embeddings. Reports overall and per-class accuracies, Precision, Recall, F1 score, ROC AUC, and displays the Confusion Matrix.

In [ ]:
# Load test embeddings
X_test = torch.tensor(np.load("test_embeddings.npy"), dtype=torch.float32)
y_test = torch.tensor(np.load("test_labels.npy"), dtype=torch.long)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE_TRAIN, shuffle=False)

test_probs = []
test_targets = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = best_model(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        test_probs.extend(probs.cpu().numpy())
        test_targets.extend(labels.numpy())

test_probs = np.array(test_probs)
test_targets = np.array(test_targets)

# Classify predictions using the optimized threshold
test_preds = (test_probs >= optimal_threshold).astype(int)

# Compute Evaluation Metrics
test_acc = accuracy_score(test_targets, test_preds)
test_precision = precision_score(test_targets, test_preds)
test_recall = recall_score(test_targets, test_preds)
test_f1 = f1_score(test_targets, test_preds, average='macro')
test_auc = roc_auc_score(test_targets, test_probs)
test_cm = confusion_matrix(test_targets, test_preds)

# Calculate Per-Class Accuracy
acc_genuine = test_cm[0, 0] / (test_cm[0, 0] + test_cm[0, 1])
acc_deepfake = test_cm[1, 1] / (test_cm[1, 0] + test_cm[1, 1])

genuine_passed = "PASSED" if acc_genuine >= 0.75 else "FAILED"
deepfake_passed = "PASSED" if acc_deepfake >= 0.75 else "FAILED"

print("\n================ TEST EVALUATION RESULTS ================")
print(f"Accuracy  : {test_acc:.4%}")
print(f"Precision : {test_precision:.4%}")
print(f"Recall    : {test_recall:.4%}")
print(f"F1 Score  : {test_f1:.4%}")
print(f"ROC AUC   : {test_auc:.4f}")
print("---------------------------------------------------------")
print(f"Genuine (Real) Accuracy  : {acc_genuine:.4%} ({genuine_passed})")
print(f"Deepfake (Fake) Accuracy : {acc_deepfake:.4%} ({deepfake_passed})")
print("=========================================================")
print("Confusion Matrix:")
print(test_cm)

## 14. Performance Visualizations

Plots training curves (Loss and Accuracy), ROC curves (highlighting the EER threshold point), and a confusion matrix heatmap.

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Train/Val Loss Curve
axes[0, 0].plot(range(1, epochs + 1), history['train_loss'], label='Train Loss', color='#1f77b4', marker='o')
axes[0, 0].plot(range(1, epochs + 1), history['val_loss'], label='Val Loss', color='#d62728', marker='x')
axes[0, 0].set_title("Train & Val Loss per Epoch", fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel("Epoch", fontsize=11)
axes[0, 0].set_ylabel("Loss", fontsize=11)
axes[0, 0].legend(fontsize=10)

# 2. Train/Val Accuracy Curve
axes[0, 1].plot(range(1, epochs + 1), history['train_acc'], label='Train Acc', color='#1f77b4', marker='o')
axes[0, 1].plot(range(1, epochs + 1), history['val_acc'], label='Val Acc', color='#d62728', marker='x')
axes[0, 1].set_title("Train & Val Accuracy per Epoch", fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel("Epoch", fontsize=11)
axes[0, 1].set_ylabel("Accuracy", fontsize=11)
axes[0, 1].legend(fontsize=10)

# 3. ROC Curve with optimal threshold label
axes[1, 0].plot(fpr, tpr, color='#ff7f0e', lw=2, label=f'ROC Curve (AUC = {val_auc:.4f})')
axes[1, 0].plot([0, 1], [0, 1], color='#7f7f7f', lw=1.5, linestyle='--')
axes[1, 0].scatter(fpr[eer_idx], tpr[eer_idx], color='red', s=80, marker='o', label=f'Optimal EER ({optimal_threshold:.4f})')
axes[1, 0].set_xlim([0.0, 1.0])
axes[1, 0].set_ylim([0.0, 1.05])
axes[1, 0].set_xlabel('False Positive Rate', fontsize=11)
axes[1, 0].set_ylabel('True Positive Rate', fontsize=11)
axes[1, 0].set_title('ROC Curve (Validation Split)', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc="lower right", fontsize=10)

# 4. Confusion Matrix Heatmap
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1], 
            xticklabels=['Real (0)', 'Fake (1)'], yticklabels=['Real (0)', 'Fake (1)'])
axes[1, 1].set_title("Confusion Matrix (Test Split)", fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel("True Class", fontsize=11)
axes[1, 1].set_xlabel("Predicted Class", fontsize=11)

plt.tight_layout()
plt.savefig("evaluation_results.png", dpi=300)
plt.show()

## 15. Saving Model & Config Metadata

Saves the trained model state dict (`best_classifier.pth`) and a JSON configuration file containing the optimal classification threshold.

In [ ]:
config_metadata = {
    "model_name": "facebook/wav2vec2-base",
    "sample_rate": TARGET_SR,
    "clip_duration": CLIP_DURATION,
    "embedding_size": 768,
    "threshold": float(optimal_threshold)
}

with open("classifier_config.json", "w") as f:
    json.dump(config_metadata, f, indent=4)

print("Model state saved: best_classifier.pth")
print("Config saved: classifier_config.json")
print("Exported details:")
print(json.dumps(config_metadata, indent=2))